# knowledge distillation with Vision Transformers


Knowledge distillation is teaching a small neural network to behave like a large powerful network.

> Big model → Teacher → Small model

The teacher already knows how to solve the task. Instead of making the student learn only from the answer key.  We also let it observe how teacher thinks about each example.

A traditional supervised learning teacher says:

> This is golden retriever

After seeing the image of dog,  thats useful but limited.

Now an expert says:

> I am 85% confident this is a golden retriever, 10% labrador Retriever and blah blah

We may know:

- Golden retriever and labrador are visually similar
- the example is probably not chihuahua
- the expert isnt relatively certain

that relative confidence contains information.

and this is the fundamental insight behind knowledge distilation.



### hard labels vs soft labels

Suppose our classes are:


```
Cat
dog
horse
car
```

A normal training label might be:

```
[0,1,0,0]
```
meaning:
> This is dog
which is called a hard target.

But teacher might output:
```
[0.05, 0.80, 0.10, 0.05]
```

Meaning:
```
  80% dog
  80% dog
  10% horse
  5% cat
  5% car
```

which is soft target..

The student therefore receives considerably richer infornmation.


Teacher is not saying dog but its saying Dog and here`s how i distinguish dog from everything else.


Thats the knowledge we want to transfer.


The teacher`s probability distribution contains information about relationships between classes.

this is sometimes called **Dark knowledge** in the knowledge-distillation literature.



Now if teacher is a **Vision Transformer**

Suppose we have a large vision Transformer:


We want a much smaller model:

**ViT-Tiny**

Our setup becomes:


```
                 Image
                   │
                   ▼
          ┌─────────────────┐
          │  Large ViT      │
          │    TEACHER      │
          └────────┬────────┘
                   │
              predictions
                   │
                   ▼
          ┌─────────────────┐
          │  Small ViT      │
          │    STUDENT      │
          └─────────────────┘

```
The Teacher is frozen .

The student learns.



### What does the student learn actually ?

Student is not copying teacher`s weight.


we are ot doing :
```
Teacher weight 1 → Student weight 1
Teacher weight 2 → Student weight 2
...

```

Instead
> when you see this image, try to produce outputs similar to teacher.

so:

```
Teacher:
Image → [0.05, 0.80, 0.10, 0.05]

Student:
Image → [0.10, 0.70, 0.15, 0.05]
```
The training objective pushes the students distributions towards the teacher.



### Mathematical heart

The student has two teachers in sense:

Teacher #1 : Ground truth

The dataset says:
```
Image → Dog
```
so we use ordinary cross entropy loss

> $L_{CE}$




Teacher #2 : The neural network teacher

The large model says:


```
Image →
dog: 0.80
horse: 0.10
cat: 0.05
car: 0.05

```

We want the student to imitate this distribution.

thats the **distilation loss**

> $L_{KD}$


$$L = \alpha L_{CE} + \beta L_{KD}$$

basically the entire idea.


### What is KL divergence doing ?

$$D_{KL}(P_T \parallel P_S)$$

This is kullback-Leibler divergence between the teacher distribution and student distribution.

> " How different is the student`s probability from the teachers?"


For example:

Teacher
```
[0.05, 0.80, 0.10, 0.05]
```

student
```
[0.04, 0.82, 0.09, 0.05]
```
Very similar → low KL divergence

but:


```
Teacher:
[0.05, 0.80, 0.10, 0.05]

Student:
[0.70, 0.10, 0.10, 0.10]

```

Very different → high KL divergence.

Training tries to make this difference small.


### Why do we use temperature ?

Nornally softmax might produce

```
[0.001,0.97,0.02,0.009]
```

Thats still relatively hard.

The smaller probabilities contains the little information.

so we introduce **Temperature T**

Conceptually:

> Temperature makes the teachers predictions softer so that hidden relationships between classes becomes easier for the student to see.


For example

**Low Temperature**

```
Dog       0.97
Horse     0.02
Cat       0.01

```

**High Temperature**

```
Dog       0.70
Horse     0.20
Cat       0.10
```

Now students gets more information about the teachers uncertainity.

way of syaing:
> "Yes, dog is the answer—but horse is somewhat similar, while cat is less similar."



### Why this matter for vision Transformer ?


Vision Transformer can be extremely powerful, but larger model require:


- more memory
- More computation
- more latency
- more energy

thats problematic when deploying CV on:

- smartphones
- drones
- Cameras
- Autonomous system
- robots
- edge GPUs
- embedded systems

Imagine:

```
Huge ViT
↓
Excellent accuracy
↓
Too slow / expensive

```
we want:

```
Small ViT
↓
Much faster
↓
Much smaller
↓
Almost as accurate
```
Knowledge distiallation attempts to bridge that gap.


### Function approximization

Let teacher represents a function:


$F(X)$

where:
- x = image
- F = Large models

we want student to learn:

$f(x)≈F(x)$

while

$f≪F$


is computational complexity.

So knowledge distillation can be viewed as;

> Approximating a powerful function with smaller function.


### why a small student learn something a label doesnot contain ?

Suppose our dataset contains:

```

Image → Golden Retriever

```
the labels only tell you

$y=Golden Retriever$


but the teacher has learned a complicated decision function.


it might implicitly understand:
```
fur texture
↓
ear shape
↓
snout
↓
body proportions
↓
color
↓
relationship to other breeds
↓
Golden Retriever

```

The label doesnot expose this structure.

The teachers output distribution gives the student additional training signal.

Thats why distiallation can outperfomr simply training the small model directly.



### Entropy


consider

**Distribution A**:

```
[0, 1, 0, 0]
```
vs

**Distribution B**

```
[0.2, 0.5, 0.2, 0.1]
```

Distribution B is more spread out.

it has higher entropy.

Roughly:

> Entropy measures how uncertain/ spread out a probability distribution is.


Low entropy:


```
████████████
```
One class dominates.


High entropy

```
████
██████
████
███

```
Probability is distributed among several possibilities.

The teacher provides a richer supervisory signal than a one hot label.

Whether entropy increase or decrease depends on:

- temperature
- callibration
- architecture
- Dataset
- training objective
- Distiallation method
- model capacity

#### Soft targets useful for gradients

Suppose the correct answer is:

```
Dog
```
cross-entropy sees:

```
dog = 1
everything else = 0
```

That creates a very coarse learning signal.

but if teacher said

```
dog    = 0.75
wolf   = 0.15
fox    = 0.08
car    = 0.02

```
now the student gets a more nuanced signal.



It learns:

> "Dog is correct, but fox and wolf are completely unrelated

that can lead to moee informative gradient updates.


so instead of learning only:

> " Dont make this mistake

the student can learn

> Here`s the direction in which your prediction should move and how strongly.


#### Distilation can use unlabeled images

Suppose:

```

1 million labeled images
+
10 million unlabeled images
```

Normally those 10 million images are difficult to use for ordinary supervised learning.

But we have a teacher:

so:

```
unlabeled image
      ↓
   Teacher
      ↓
pseudo-label / soft prediction
      ↓
   Student

```

The teacher effectively creates supervision

This connects knowledge distillation to semi-supervised learning and pseudo labeling.


But, Teacher isnt necessiarly correct:

```
Teacher = 95% accurate

```
The remaining 5% of predictions can contain errors

if the student blindly copies the teacher:

```
Teacher mistake
      ↓
Student learns mistake

```

so distillation isnot a magic.

We have to ask:

> What knowledge should be student inherit, and what knowledge should it reject ?




Level 1 - Answer distillation

Teacher:
> DOg

student learns
> Dog



level 2 - Probability distribution

Teacher:
> "80% dog, 15% wolf, 5% fox."

student learns the probability structure


Level 3 - Feature distillation

Teacher:

> This image has these internl visual characterstics

student tries to reproduce useful representations



Level 4 - Attention distillation

Teacher:

> These image patches are important and these patches interact

Student learns something about the teachers attention structure


Level 5-  Relational distillation

Teacher:

> These examples/features have these relationships with each other.


student learns the gemometry/relaionships of the teachers representation space.


> A good distillation system should distinguish "teacher is confident because it knows" from "teacher is confident because it is confidently wrong."



```
                 BIG MODEL
                  Teacher
                     │
        ┌────────────┼─────────────┐
        │            │             │
      logits       features     attention
        │            │             │
        └────────────┼─────────────┘
                     ↓
              DISTILLATION
                     ↓
                SMALL MODEL
                  Student
                     ↓
        ┌────────────┼─────────────┐
        │            │             │
      faster       smaller       cheaper
                     ↓
              EDGE DEPLOYMENT

```
Its about learning to approximate the teachers behaviour.

A hard label tells a model what the answer is. A teacher model can tell a student much more about why that answer is plausible relative to other answers. Knowledge distillation tries to transfer that richer information into a smaller model.



"What constitutes knowledge in a neural network, and how can I transfer the useful parts of that knowledge without transferring the cost?"


$$L = \alpha L_{CE} + \beta L_{KD}$$
$$ \boxed{ \text{Student learns from reality + teacher's beliefs} } $$